In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from functools import reduce

### Reusable Data Cleaning Functions (Keep for All Tables)
Timestamp Cleaning Function

In [0]:
def clean_timestamp(df, column):
    return df.withColumn(
        column,
        to_timestamp(
            when(col(column).isin("not_available","unknown","None",""), None)
            .otherwise(col(column))
        )
    )

In [0]:
def trim_all_strings(df):
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        
    return df

In [0]:
def trim_all_strings(df):
    string_cols = [f.name for f in df.schema.fields if f.dataType.simpleString() == "string"]
    
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        
    return df

In [0]:
df_order_items_raw = spark.table("main_workspace.bronze.order_items")

display(df_order_items_raw)

df_order_items_raw.printSchema()

print("Total rows:", df_order_items_raw.count())

In [0]:
df_trim = trim_all_strings(df_order_items_raw)

display(df_trim)

STEP 1 — Convert Empty Strings → NULL

In [0]:
df_clean = df_trim

string_cols = [
    f.name for f in df_clean.schema.fields
    if f.dataType.simpleString() == "string"
]

for c in string_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

display(df_clean)

STEP 2 — Check NULL Counts

In [0]:
display(
df_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_clean.columns
])
)

STEP 3 — Build Quarantine Condition

In [0]:
conditions = []

for c in df_clean.columns:
    conditions.append(col(c).isNull())

from functools import reduce

quarantine_condition = reduce(lambda x, y: x | y, conditions)

In [0]:
order_items_quarantine = df_clean.filter(quarantine_condition)

display(order_items_quarantine)

print("Quarantine rows:", order_items_quarantine.count())

In [0]:
valid_order_items = df_clean.subtract(order_items_quarantine)

display(valid_order_items)

print("Valid rows:", valid_order_items.count())

In [0]:
order_items_quarantine = order_items_quarantine.withColumn(
    "shipping_limit_date",
    when(col("shipping_limit_date") == "not_available", None)
    .otherwise(col("shipping_limit_date"))
)

valid_order_items = valid_order_items.withColumn(
    "shipping_limit_date",
    when(col("shipping_limit_date") == "not_available", None)
    .otherwise(col("shipping_limit_date"))
)

In [0]:
order_items_quarantine_cast = order_items_quarantine \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("order_item_id", col("order_item_id").cast("int")) \
.withColumn("product_id", col("product_id").cast("string")) \
.withColumn("seller_id", col("seller_id").cast("string")) \
.withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
.withColumn("price", col("price").cast("double")) \
.withColumn("freight_value", col("freight_value").cast("double")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
order_items_quarantine_cast.printSchema()

display(order_items_quarantine_cast)

In [0]:
order_items_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.order_items_quarantine_data")

In [0]:
valid_order_items_cast = valid_order_items \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("order_item_id", col("order_item_id").cast("int")) \
.withColumn("product_id", col("product_id").cast("string")) \
.withColumn("seller_id", col("seller_id").cast("string")) \
.withColumn("shipping_limit_date", to_timestamp(col("shipping_limit_date"))) \
.withColumn("price", col("price").cast("double")) \
.withColumn("freight_value", col("freight_value").cast("double")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(
valid_order_items_cast.select([
count(when(col(c).isNull(), c)).alias(c)
for c in valid_order_items_cast.columns
])
)

In [0]:
valid_order_items_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.order_items")

# Payments table

In [0]:
from pyspark.sql.functions import *
from functools import reduce

df_payments_raw = spark.table("main_workspace.bronze.payments")

display(df_payments_raw)

print("Total records:", df_payments_raw.count())

df_payments_raw.printSchema()

In [0]:
df_trim = df_payments_raw

string_cols = [
    f.name for f in df_trim.schema.fields
    if f.dataType.simpleString() == "string"
]

for c in string_cols:
    df_trim = df_trim.withColumn(c, trim(col(c)))

display(df_trim)

In [0]:
df_clean = df_trim

for c in string_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

display(df_clean)

Validate NULL Counts

In [0]:
display(
df_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_clean.columns
])
)

STEP 5 — Build Quarantine Condition

In [0]:
conditions = []

for c in df_clean.columns:
    conditions.append(col(c).isNull())

quarantine_condition = reduce(lambda x, y: x | y, conditions)

Extract Quarantine Rows

In [0]:
payments_quarantine = df_clean.filter(quarantine_condition)

display(payments_quarantine)

print("Quarantine rows:", payments_quarantine.count())

Extract Valid Rows

In [0]:
valid_payments = df_clean.subtract(payments_quarantine)

display(valid_payments)

print("Valid rows:", valid_payments.count())

In [0]:
payments_quarantine_cast = payments_quarantine \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("payment_sequential", col("payment_sequential").cast("int")) \
.withColumn("payment_type", col("payment_type").cast("string")) \
.withColumn("payment_installments", col("payment_installments").cast("int")) \
.withColumn("payment_value", col("payment_value").cast("double")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
payments_quarantine_cast.printSchema()

display(payments_quarantine_cast)

In [0]:
payments_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.payments_quarantine_data")

In [0]:
valid_payments_cast = valid_payments \
.withColumn("order_id", col("order_id").cast("string")) \
.withColumn("payment_sequential", col("payment_sequential").cast("double").cast("int")) \
.withColumn("payment_type", col("payment_type").cast("string")) \
.withColumn("payment_installments", col("payment_installments").cast("double").cast("int")) \
.withColumn("payment_value", col("payment_value").cast("double")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(
valid_payments_cast.select([
count(when(col(c).isNull(), c)).alias(c)
for c in valid_payments_cast.columns
])
)

In [0]:
valid_payments_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.payments")

# PRODUCTS TABLE — SILVER PIPELINE

In [0]:
from pyspark.sql.functions import *
from functools import reduce

df_products_raw = spark.table("main_workspace.bronze.products")

display(df_products_raw)

print("Total records:", df_products_raw.count())

df_products_raw.printSchema()

## Trim Whitespace

In [0]:
df_trim = df_products_raw

string_cols = [
    f.name for f in df_trim.schema.fields
    if f.dataType.simpleString() == "string"
]

for c in string_cols:
    df_trim = df_trim.withColumn(c, trim(col(c)))

display(df_trim)

Convert Empty Strings → NULL

In [0]:
df_clean = df_trim

for c in string_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

display(df_clean)

Check NULL Counts

In [0]:
display(
df_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_clean.columns
])
)

Build Quarantine Condition

In [0]:
conditions = []

for c in df_clean.columns:
    conditions.append(col(c).isNull())

quarantine_condition = reduce(lambda x, y: x | y, conditions)

Extract Quarantine Rows

In [0]:
products_quarantine = df_clean.filter(quarantine_condition)

display(products_quarantine)

print("Quarantine rows:", products_quarantine.count())

Extract Valid Rows

In [0]:
valid_products = df_clean.subtract(products_quarantine)

display(valid_products)

print("Valid rows:", valid_products.count())

Cast Quarantine Dataset

In [0]:
products_quarantine_cast = products_quarantine \
.withColumn("product_id", col("product_id").cast("string")) \
.withColumn("product_category_name", col("product_category_name").cast("string")) \
.withColumn("product_name_lenght", col("product_name_lenght").cast("double").cast("int")) \
.withColumn("product_description_lenght", col("product_description_lenght").cast("double").cast("int")) \
.withColumn("product_photos_qty", col("product_photos_qty").cast("double").cast("int")) \
.withColumn("product_weight_g", col("product_weight_g").cast("double").cast("int")) \
.withColumn("product_length_cm", col("product_length_cm").cast("double").cast("int")) \
.withColumn("product_height_cm", col("product_height_cm").cast("double").cast("int")) \
.withColumn("product_width_cm", col("product_width_cm").cast("double").cast("int")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
products_quarantine_cast.printSchema()

display(products_quarantine_cast)

In [0]:
products_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.products_quarantine_data")

In [0]:
valid_products_cast = valid_products \
.withColumn("product_id", col("product_id").cast("string")) \
.withColumn("product_category_name", col("product_category_name").cast("string")) \
.withColumn("product_name_lenght", col("product_name_lenght").cast("double").cast("int")) \
.withColumn("product_description_lenght", col("product_description_lenght").cast("double").cast("int")) \
.withColumn("product_photos_qty", col("product_photos_qty").cast("double").cast("int")) \
.withColumn("product_weight_g", col("product_weight_g").cast("double").cast("int")) \
.withColumn("product_length_cm", col("product_length_cm").cast("double").cast("int")) \
.withColumn("product_height_cm", col("product_height_cm").cast("double").cast("int")) \
.withColumn("product_width_cm", col("product_width_cm").cast("double").cast("int")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(
valid_products_cast.select([
count(when(col(c).isNull(), c)).alias(c)
for c in valid_products_cast.columns
])
)

In [0]:
valid_products_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.products")

# PRODUCT_CATEGORY_TRANSLATION — SILVER PIPELINE

In [0]:
from pyspark.sql.functions import *
from functools import reduce

df_category_raw = spark.table("main_workspace.bronze.product_category_translation")

display(df_category_raw)

print("Total records:", df_category_raw.count())

df_category_raw.printSchema()

In [0]:
df_trim = df_category_raw

string_cols = [
    f.name for f in df_trim.schema.fields
    if f.dataType.simpleString() == "string"
]

for c in string_cols:
    df_trim = df_trim.withColumn(c, trim(col(c)))

display(df_trim)

In [0]:
df_clean = df_trim

for c in string_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

display(df_clean)

In [0]:
display(
df_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_clean.columns
])
)

In [0]:
conditions = []

for c in df_clean.columns:
    conditions.append(col(c).isNull())

quarantine_condition = reduce(lambda x, y: x | y, conditions)

In [0]:
category_quarantine = df_clean.filter(quarantine_condition)

display(category_quarantine)

print("Quarantine rows:", category_quarantine.count())

In [0]:
valid_category = df_clean.subtract(category_quarantine)

display(valid_category)

print("Valid rows:", valid_category.count())

In [0]:
valid_category_cast = valid_category \
.withColumn("product_category_name", col("product_category_name").cast("string")) \
.withColumn("product_category_name_english", col("product_category_name_english").cast("string")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(
valid_category_cast.select([
count(when(col(c).isNull(), c)).alias(c)
for c in valid_category_cast.columns
])
)

In [0]:
valid_category_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.product_category_translation")

# SELLERS TABLE — SILVER PIPELINE

In [0]:
from pyspark.sql.functions import *
from functools import reduce

df_sellers_raw = spark.table("main_workspace.bronze.sellers")

display(df_sellers_raw)

print("Total records:", df_sellers_raw.count())

df_sellers_raw.printSchema()

In [0]:
df_trim = df_sellers_raw

string_cols = [
    f.name for f in df_trim.schema.fields
    if f.dataType.simpleString() == "string"
]

for c in string_cols:
    df_trim = df_trim.withColumn(c, trim(col(c)))

display(df_trim)

In [0]:
df_clean = df_trim

for c in string_cols:
    df_clean = df_clean.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

display(df_clean)

In [0]:
display(
df_clean.select([
count(when(col(c).isNull(), c)).alias(c)
for c in df_clean.columns
])
)

In [0]:
conditions = []

for c in df_clean.columns:
    conditions.append(col(c).isNull())

quarantine_condition = reduce(lambda x, y: x | y, conditions)

In [0]:
sellers_quarantine = df_clean.filter(quarantine_condition)

display(sellers_quarantine)

print("Quarantine rows:", sellers_quarantine.count())

In [0]:
valid_sellers = df_clean.subtract(sellers_quarantine)

display(valid_sellers)

print("Valid rows:", valid_sellers.count())

In [0]:
sellers_quarantine_cast = sellers_quarantine \
.withColumn("seller_id", col("seller_id").cast("string")) \
.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("double").cast("int")) \
.withColumn("seller_city", col("seller_city").cast("string")) \
.withColumn("seller_state", col("seller_state").cast("string")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("QUARANTINE"))

In [0]:
sellers_quarantine_cast.printSchema()

display(sellers_quarantine_cast)

In [0]:
sellers_quarantine_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.quarantine_data.sellers_quarantine_data")

In [0]:
valid_sellers_cast = valid_sellers \
.withColumn("seller_id", col("seller_id").cast("string")) \
.withColumn("seller_zip_code_prefix", col("seller_zip_code_prefix").cast("double").cast("int")) \
.withColumn("seller_city", col("seller_city").cast("string")) \
.withColumn("seller_state", col("seller_state").cast("string")) \
.withColumn("processed_timestamp", current_timestamp()) \
.withColumn("data_quality_flag", lit("VALID"))

In [0]:
display(
valid_sellers_cast.select([
count(when(col(c).isNull(), c)).alias(c)
for c in valid_sellers_cast.columns
])
)

In [0]:
valid_sellers_cast.write \
.format("delta") \
.mode("overwrite") \
.saveAsTable("main_workspace.silver.sellers")